In [1]:
from pathlib import Path
import os
import time
import sys
repo_dir =  Path.cwd()
os.environ["HF_HOME"] = str(repo_dir / ".hf_cache")
sys.path.insert(0, str(repo_dir / "src"))
from lp_llm.solver import solve_lp_problem
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM



<python-env>\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
print(f"VRAM       : {torch.cuda.memory_allocated() / 1024**3:.2f} Go")
print(f"Reserved VRAM       : {torch.cuda.memory_reserved() / 1024**3:.2f} Go")
a,b = torch.cuda.mem_get_info()
print(f"Free VRAM  : {a/1024**3:.2f} Go")
print(f"Total VRAM : {b/1024**3:.2f} Go")

VRAM       : 0.00 Go
Reserved VRAM       : 0.00 Go
Free VRAM  : 6.94 Go
Total VRAM : 8.00 Go


In [3]:
model_id = "Qwen/Qwen2.5-1.5B-Instruct"
t0 = time.perf_counter()

tokenizer = AutoTokenizer.from_pretrained(model_id,revision="989aa7980e4cf806f80c7fef2b1adb7bc71aa306")

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.bfloat16,
    device_map="auto",
    revision="989aa7980e4cf806f80c7fef2b1adb7bc71aa306"
)

print(model.dtype)  # torch.bfloat16
model.eval()

torch.cuda.synchronize()
loading_time = time.perf_counter() - t0

print(f"Chargement    : {loading_time:.2f} s")
print(f"Device        : {model.device}")
print(f"Dtype         : {model.dtype}")

device_map = getattr(model, "hf_device_map", None)

if device_map is not None:
    print(f"hf_device_map : {device_map}")
else:
    print(f"Devices des paramètres : {set(p.device for p in model.parameters())}")

a,b = torch.cuda.mem_get_info()
print(f"Free VRAM     : {a/1024**3:.2f} Go")
print(f"Total VRAM    : {b/1024**3:.2f} Go")
print(f"VRAM          : {torch.cuda.memory_allocated() / 1024**3:.2f} Go")
print(f"Reserved      : {torch.cuda.memory_reserved()/1024**3:.2f} Go")

W1106 13:22:03.496000 38256 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
Loading weights: 100%|██████████| 338/338 [00:16<00:00, 20.21it/s]

torch.bfloat16
Chargement    : 24.41 s
Device        : cuda:0
Dtype         : torch.bfloat16
Devices des paramètres : {device(type='cuda', index=0)}
Free VRAM     : 4.01 Go
Total VRAM    : 8.00 Go
VRAM          : 2.88 Go
Reserved      : 2.93 Go


In [4]:
n_params = sum(p.numel() for p in model.parameters())
print(f"Paramètres : {n_params:,} ({n_params / 1e9:.2f} B)")

# Dtype et device
p = next(model.parameters())
print(f"Dtype      : {p.dtype}")
print(f"Device     : {p.device}")

# Configuration du modèle
config = model.config
print(f"Couches    : {config.num_hidden_layers}")
print(f"Hidden size: {config.hidden_size}")
print(f"Contexte max : {config.max_position_embeddings}")

# Tokenizer
print(f"Vocabulaire : {len(tokenizer)}")
print(f"EOS token   : {tokenizer.eos_token} ({tokenizer.eos_token_id})")
print(f"PAD token   : {tokenizer.pad_token} ({tokenizer.pad_token_id})")

Paramètres : 1,543,714,304 (1.54 B)
Dtype      : torch.bfloat16
Device     : cuda:0
Couches    : 28
Hidden size: 1536
Contexte max : 32768
Vocabulaire : 151665
EOS token   : <|im_end|> (151645)
PAD token   : <|endoftext|> (151643)


In [5]:
messages = [ 
    {"role": "system", 
    "content": """
    You are producing a json, no more, no less. 
    It should represent a linear problem in the form of for example : json is {
    "variables": ["x1", "x2"],
    "objective": {"sense": "min", "coefficients": [5, 7.5]},
    "constraints": [
        {"coefficients": [1, 1], "sense": "<=", "rhs": 3},
        {"coefficients": [1, 1], "sense": "<=", "rhs": 0.5}
    ],
    "bounds": [[0, null], [0.1, 12]]
}.
Constraints on the way to write the problem : 
 1 - Always 2 variables, named x1 and x2. 
 2 - json["objective"]["coefficients"] must be a list of length 2 (because 2 variables).
 3 -  json["objective"]["sense"] is in {"min","max"}
 4 - json["constraints"] must be a list of length 2 or 3
 5 - for all i, json["constraints"][i]["coefficients"] must be a list of length 2 (because 2 variables).
 6 - for all i, json["constraints"][i]["sense"] is in {"<=","=",">="}
 7 - json["bounds"] must be a list of length 2 (because 2 variables).
 8 - for all i, json["bounds"][i] must be a list of length 2
 9 - for all i, json["bounds"][i][0] can't be negative
 10 - for all i, json["bounds"][i][0]<= json["bounds"][i][1]
 11 - "coefficients", "rhs" and elements of bounds should be numbers ( int or float)
 12 - for json["bounds"][i][j], null represent no bound, ie +infinity
 13 - sometimes "coefficients" and "rhs" can be presented to you with different units, e.g. hour and minutes, convert, and put results directly in the json.
 
 forbidden characters : ``` or //
 Answer only with the valid JSON (all your generated text could pass a json.loads), without text or markdown .
"""}, 
{"role": "user", "content": """
A bakery produces two types of products: **cakes and cookies**.

The bakery has a daily supply of **20 kg of flour** and a maximum of **8 working hours** available each day.

Each product requires different resources and generates a different profit:

| | Cakes | Cookies |
|---|---|---|
| Flour required per unit | 500 g | 200 g |
| Preparation time per unit | 30 minutes | 10 minutes |
| Profit per unit | €12 | €4 |

**Objective:** Maximize the bakery's total daily profit.

**Decision variables:**
- The number of cakes produced each day.
- The number of cookies produced each day.

**Constraints:**
1. **Flour availability:** The total amount of flour used cannot exceed 20 kg per day.
2. **Working time:** The total preparation time cannot exceed 8 hours per day.

**Question:** How many cakes and cookies should the bakery produce each day to maximize its profit while respecting both constraints?
"""
}]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(prompt)
inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True,return_tensors="pt",return_dict=True).to(model.device)
#print(len(inputs))
print(50*'-')
# Génération
with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False
    )
    
    input_len = inputs["input_ids"].shape[-1]   # nombre de tokens du prompt
    new_tokens = outputs[0, input_len:]          # uniquement les tokens générés
    last = new_tokens[-1].item()                 # id du dernier token, en entier Python

print(f"Nouveaux tokens : {len(new_tokens)} / max 256")
print(f"Dernier token   : {last} -> {tokenizer.decode([last])!r}")
print(f"Fin par EOS     : {last == tokenizer.eos_token_id}")
print(50*'-')
# Décodage : uniquement les nouveaux tokens
response = tokenizer.decode(
    outputs[0, inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True
)

print(response)

solve_lp_problem(response)

<|im_start|>system

    You are producing a json, no more, no less. 
    It should represent a linear problem in the form of for example : json is {
    "variables": ["x1", "x2"],
    "objective": {"sense": "min", "coefficients": [5, 7.5]},
    "constraints": [
        {"coefficients": [1, 1], "sense": "<=", "rhs": 3},
        {"coefficients": [1, 1], "sense": "<=", "rhs": 0.5}
    ],
    "bounds": [[0, null], [0.1, 12]]
}.
Constraints on the way to write the problem : 
 1 - Always 2 variables, named x1 and x2. 
 2 - json["objective"]["coefficients"] must be a list of length 2 (because 2 variables).
 3 -  json["objective"]["sense"] is in {"min","max"}
 4 - json["constraints"] must be a list of length 2 or 3
 5 - for all i, json["constraints"][i]["coefficients"] must be a list of length 2 (because 2 variables).
 6 - for all i, json["constraints"][i]["sense"] is in {"<=","=",">="}
 7 - json["bounds"] must be a list of length 2 (because 2 variables).
 8 - for all i, json["bounds"][i] must

{'status': 'invalid',
 'message': 'Invalid JSON: Expecting value: line 1 column 1 (char 0)'}

In [6]:
#test du solveur sur JSON corrigé à la main, pas une sortie du modèle
lp_json = """
{
  "variables": ["x1", "x2"],
  "objective": {"sense": "max", "coefficients": [12, 4]},
  "constraints": [
    {"coefficients": [0.5, 0.2], "sense": "<=", "rhs": 20},
    {"coefficients": [30, 10], "sense": "<=", "rhs": 480} 
  ],
  "bounds": [[0, null], [0, null]]
}
"""
solve_lp_problem(lp_json)

{'status': 'optimal',
 'message': 'Optimal solution found.',
 'solution': [16.0, 0.0],
 'objective_value': 192.0}